In [15]:
from pathlib import Path
import pandas as pd

AUDIO_ROOT = Path("/Users/alicia/Library/CloudStorage/GoogleDrive-adaniell@usc.edu/My Drive/CSCI567 Project/modma_data/audio_lanzhou_2015")
DROP_COLS = ("file", "start", "end")  # openSMILE metadata we don't want as features

func_dfs: dict[int, pd.DataFrame] = {}
subject_dirs = sorted(p for p in AUDIO_ROOT.iterdir() if p.is_dir())

for file_idx in range(1, 30):
    stem = f"{file_idx:02d}"
    rows: {}
    feature_cols: None

    for subj_dir in subject_dirs:
        func_csv = subj_dir / f"{stem}_openSMILE_func.csv"
        if not func_csv.exists():
            continue
        df = pd.read_csv(func_csv)
        df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

        if feature_cols is None:
            feature_cols = list(df.columns)
        elif list(df.columns) != feature_cols:
            raise ValueError(
                f"Column mismatch in {func_csv}:\n"
                f"  expected {feature_cols[:3]}...\n"
                f"  got      {list(df.columns)[:3]}..."
            )

        rows[subj_dir.name] = df.iloc[0]

    out = pd.DataFrame.from_dict(rows, orient="index", columns=feature_cols)
    out.index.name = "subject_id"
    func_dfs[file_idx] = out

# Sanity check — should print real feature names, not 0..87
print(func_dfs[1].columns.tolist()[:5])
print(func_dfs[1].shape)

['F0semitoneFrom27.5Hz_sma3nz_amean', 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'F0semitoneFrom27.5Hz_sma3nz_percentile20.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile50.0', 'F0semitoneFrom27.5Hz_sma3nz_percentile80.0']
(52, 88)


In [16]:
func_dfs[1]

,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope,F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope,F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
subject_id,,,,,,,,,,,,,,,,,,,,,
02010001,24.908050,0.076967,23.269852,25.082000,26.457945,3.188093,159.477310,147.329570,42.670486,49.601814,...,-0.047516,-0.010459,0.014082,1.601602,1.309164,0.153846,0.112424,0.636667,0.497868,-55.350494
02010002,35.676167,0.168829,27.614733,38.846783,39.607388,11.992655,368.519440,413.960100,65.847220,46.331623,...,-0.057903,-0.006957,0.021004,2.275250,1.279911,0.189565,0.135950,0.522800,0.617262,-55.456734
02010003,28.712692,0.144249,26.646698,28.423923,30.694073,4.047375,980.981260,1034.272100,33.911580,18.169123,...,-0.055786,0.001762,0.025030,2.452620,2.354260,0.212857,0.174742,0.190952,0.256996,-45.951240
02010004,33.834644,0.232009,30.321724,36.880585,39.416508,9.094784,155.532200,149.551400,56.141790,15.768618,...,-0.057733,-0.006071,0.019145,1.494565,1.369863,0.217000,0.106494,0.491000,0.557215,-46.021570
02010005,22.789120,0.074689,21.288550,22.544037,24.234898,2.946348,102.627640,124.026300,19.537884,9.936992,...,-0.063791,-0.005362,0.018774,3.477499,1.668130,0.182456,0.152146,0.403929,0.510471,-55.470848
02010006,28.373934,0.100747,26.071209,28.154602,31.076426,5.005217,267.566280,312.282930,31.619465,22.968718,...,-0.051852,-0.013608,0.021173,3.299293,1.657458,0.257619,0.180737,0.358947,0.328295,-48.761616
02010008,22.900763,0.174059,20.159597,22.668007,25.058414,4.898817,308.449740,370.611480,41.038147,46.111553,...,-0.057620,-0.005867,0.022838,2.000000,1.472393,0.211389,0.147048,0.447222,0.593954,-46.237293
02010009,20.629217,0.088746,19.411980,20.587120,22.514462,3.102482,31.684850,9.132483,33.578842,9.993665,...,-0.060720,-0.002252,0.021841,3.208556,1.648352,0.146667,0.103387,0.327500,0.259073,-53.708088
02010010,25.347515,0.126585,23.024849,24.663824,27.669550,4.644701,49.110180,25.130816,21.957525,15.527277,...,-0.073213,-0.001169,0.030454,2.599010,1.867995,0.318667,0.217251,0.171176,0.227179,-47.524980


In [17]:
# CELL 3 — Load the subject-info and audio-file-map CSVs.
#
# subject_info:   one row per audio subject (52 rows).
#                 Columns: group, age, gender, edu_years, PHQ-9,
#                          CTQ-SF, LES, SSRS, GAD-7, PSQI.
#                 Indexed by subject_id (zero-padded 8-digit string).
# audio_file_map: one row per wav file number 1..29 (29 rows).
#                 Columns: task, task_number, valence, notes.

from pathlib import Path
import pandas as pd

CSV_ROOT = Path("/Users/alicia/Library/CloudStorage/GoogleDrive-adaniell@usc.edu/My Drive/CSCI567 Project")

subject_info = pd.read_csv(
    CSV_ROOT / "subject_info_map.csv",
    dtype={"subject_id": str},
).set_index("subject_id")

audio_file_map = pd.read_csv(CSV_ROOT / "audio_file_map.csv")

print(f"subject_info:    {subject_info.shape}  "
      f"groups={subject_info['group'].value_counts().to_dict()}")
print(f"audio_file_map:  {audio_file_map.shape}  "
      f"tasks={audio_file_map['task'].value_counts().to_dict()}")
print(f"PHQ-9 — MDD range {subject_info.loc[subject_info.group=='MDD', 'PHQ-9'].min()}"
      f"–{subject_info.loc[subject_info.group=='MDD', 'PHQ-9'].max()}, "
      f"HC range {subject_info.loc[subject_info.group=='HC', 'PHQ-9'].min()}"
      f"–{subject_info.loc[subject_info.group=='HC', 'PHQ-9'].max()}")
subject_info.head(3)


subject_info:    (52, 10)  groups={'HC': 29, 'MDD': 23}
audio_file_map:  (29, 5)  tasks={'Interview': 18, 'Word Reading': 6, 'Picture Description': 4, 'Passage Reading': 1}
PHQ-9 — MDD range 6–25, HC range 0–9


,group,age,gender,edu_years,PHQ-9,CTQ-SF,LES,SSRS,GAD-7,PSQI
subject_id,,,,,,,,,,
02010002,MDD,18,F,12,23,77,-143,31,18,12
02010004,MDD,25,F,19,12,53,-44,38,13,11
02010005,MDD,20,M,16,19,49,-3,28,11,5


In [ ]:
# CELL 4 — Aggregate per-subject mean across the 29 openSMILE files.
#
# Stack all 29 (52 x 88) DataFrames in func_dfs into a tall frame, then
# group-by subject_id and take the mean. Subjects with missing files
# (e.g. 02010004 is missing 5 corrupt wavs) are averaged over whatever
# files they do have.

stacked = pd.concat(func_dfs.values(), axis=0)
subject_mean_88 = stacked.groupby(level=0).mean()
subject_mean_88.index.name = "subject_id"

# Align to subject_info row order (should already match)
subject_mean_88 = subject_mean_88.loc[subject_info.index]

files_per_subject = stacked.groupby(level=0).size()
print(f"Files per subject — min {files_per_subject.min()}, "
      f"median {int(files_per_subject.median())}, "
      f"max {files_per_subject.max()}")
missing = files_per_subject[files_per_subject < 29]
if len(missing):
    print(f"Subjects with <29 files: {missing.to_dict()}")
else:
    print("All subjects have all 29 files.")

print(f"\nsubject_mean_88 shape: {subject_mean_88.shape}")
subject_mean_88.head(3)


In [ ]:
# CELL 5 — 80/20 train/test split, STRATIFIED by MDD/HC group.
#
# Should we stratify? YES.
#
#  (a) n=52 with 23 MDD / 29 HC (44% / 56%). A random unstratified
#      80/20 split leaves ~10 test subjects; class composition of
#      that test set could range from 1 to 7 MDD purely by chance.
#  (b) Stratified CV/splitting reduces estimate variance and bias
#      at small n (Kohavi 1995, IJCAI, "A Study of Cross-Validation
#      and Bootstrap for Accuracy Estimation and Model Selection").
#  (c) PHQ-9 is bimodal. An unstratified test set could be
#      dominated by one mode, yielding a misleading RMSE that
#      doesn't reflect performance across the target range.
#  (d) Standard practice in clinical-speech depression work
#      (Cummins et al. 2015 Speech Communication review;
#      Low et al. 2020 Laryngoscope Investig. Otolaryngol. review).
#
# Alternative: stratify by binned PHQ-9. Overkill for a 10-point
# test set — group-label stratification covers the bimodality.

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_all = subject_mean_88.values                  # (52, 88)
y_all = subject_info["PHQ-9"].values            # (52,)
groups_all = subject_info["group"].values       # (52,) — MDD or HC
idx_all = subject_info.index.values             # subject_ids

X_train, X_test, y_train, y_test, grp_train, grp_test, idx_train, idx_test = train_test_split(
    X_all, y_all, groups_all, idx_all,
    test_size=0.20, stratify=groups_all, random_state=RANDOM_STATE,
)

print(f"Train: X={X_train.shape}  groups={pd.Series(grp_train).value_counts().to_dict()}  "
      f"PHQ-9 range {y_train.min()}-{y_train.max()}")
print(f"Test : X={X_test.shape}   groups={pd.Series(grp_test).value_counts().to_dict()}  "
      f"PHQ-9 range {y_test.min()}-{y_test.max()}")
print(f"\nTest subject_ids: {list(idx_test)}")


In [ ]:
# CELL 6 — LOSO Elastic Net on the TRAINING set.
#
# Structure:
#  1. Fit StandardScaler on TRAIN ONLY and apply to train+test (no leakage).
#  2. ElasticNetCV with LeaveOneOut inner CV selects (alpha, l1_ratio) on
#     training data. This is the "LOSO" part — each training subject is
#     held out in turn during hyperparameter search.
#  3. ElasticNetCV refits the final model on the full training set at the
#     chosen hyperparameters.
#  4. Separately, generate training LOSO out-of-fold predictions via
#     cross_val_predict for diagnostic plotting in Cell 8.

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.model_selection import LeaveOneOut, cross_val_predict

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

loo = LeaveOneOut()

enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
    alphas=np.logspace(-3, 1, 40),
    cv=loo,
    max_iter=20000,
    n_jobs=-1,
    random_state=RANDOM_STATE,
).fit(X_train_s, y_train)

n_nonzero = int((enet_cv.coef_ != 0).sum())
print(f"ElasticNetCV — chosen hyperparameters:")
print(f"  alpha (lambda) = {enet_cv.alpha_:.4f}")
print(f"  l1_ratio       = {enet_cv.l1_ratio_:.2f}  "
      f"(1.0=pure LASSO, 0.0=pure Ridge)")
print(f"  nonzero coefs  = {n_nonzero} / {len(enet_cv.coef_)}")

# Training LOSO out-of-fold predictions at the chosen hyperparameters
final_enet = ElasticNet(
    alpha=enet_cv.alpha_, l1_ratio=enet_cv.l1_ratio_,
    max_iter=20000, random_state=RANDOM_STATE,
)
loso_pred_train = cross_val_predict(final_enet, X_train_s, y_train, cv=loo)
print(f"\nloso_pred_train shape: {loso_pred_train.shape}  "
      f"(one out-of-fold prediction per training subject)")


In [ ]:
# CELL 7 — Per-test-subject loss and overall metrics.
# Also compare against training LOSO-OOF performance (checks for overfit
# to the outer split) and a trivial mean-predictor baseline.

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_test = enet_cv.predict(X_test_s)

test_results = pd.DataFrame({
    "subject_id": idx_test,
    "group": grp_test,
    "PHQ9_actual": y_test,
    "PHQ9_pred": y_pred_test,
    "abs_error": np.abs(y_test - y_pred_test),
    "sq_error": (y_test - y_pred_test) ** 2,
}).set_index("subject_id").sort_values("abs_error", ascending=False)

print("Per-test-subject predictions (sorted by abs_error desc):")
print(test_results.round(3))

print("\n--- Test-set metrics ---")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.3f}")
print(f"  MAE : {mean_absolute_error(y_test, y_pred_test):.3f}")
print(f"  R^2 : {r2_score(y_test, y_pred_test):.3f}")

print("\n--- Training LOSO-OOF metrics (diagnostic) ---")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, loso_pred_train)):.3f}")
print(f"  MAE : {mean_absolute_error(y_train, loso_pred_train):.3f}")
print(f"  R^2 : {r2_score(y_train, loso_pred_train):.3f}")

baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
print(f"\nMean-predictor baseline test RMSE: {np.sqrt(mean_squared_error(y_test, baseline_pred)):.3f}")
print(f"(If Elastic Net test RMSE >= baseline, the model adds no value.)")


In [ ]:
# CELL 8 — Visualization.
# Left panel:  predicted vs actual PHQ-9. Train points = LOSO out-of-fold,
#              test points = true held-out (shown as stars). Colored by
#              MDD/HC group. Dashed y=x line shows perfect prediction.
#              Gray band marks the empirical PHQ-9 dead zone (7–10).
# Right panel: per-test-subject residuals (pred - actual), labeled with
#              subject_id, group, and actual PHQ-9, sorted by actual.

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel 1: predicted vs actual
ax = axes[0]
train_is_mdd = (grp_train == "MDD")
test_is_mdd = (grp_test == "MDD")

ax.scatter(y_train[~train_is_mdd], loso_pred_train[~train_is_mdd],
           s=45, alpha=0.5, c="tab:blue", marker="o", label="Train HC (LOSO-OOF)")
ax.scatter(y_train[train_is_mdd], loso_pred_train[train_is_mdd],
           s=45, alpha=0.5, c="tab:red", marker="o", label="Train MDD (LOSO-OOF)")
ax.scatter(y_test[~test_is_mdd], y_pred_test[~test_is_mdd],
           s=200, alpha=0.95, c="tab:blue", marker="*",
           edgecolors="black", linewidths=1.2, label="Test HC")
ax.scatter(y_test[test_is_mdd], y_pred_test[test_is_mdd],
           s=200, alpha=0.95, c="tab:red", marker="*",
           edgecolors="black", linewidths=1.2, label="Test MDD")

lims = [-2, 28]
ax.plot(lims, lims, "k--", alpha=0.4, label="y = x (perfect)")
ax.axvspan(7, 11, color="gray", alpha=0.12, label="PHQ-9 dead zone")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual PHQ-9")
ax.set_ylabel("Predicted PHQ-9")
ax.set_title("LOSO Elastic Net — predicted vs actual PHQ-9")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

# --- Panel 2: per-test-subject residuals
ax = axes[1]
residuals_df = test_results.sort_values("PHQ9_actual")
colors = ["tab:red" if g == "MDD" else "tab:blue" for g in residuals_df["group"]]
resid = residuals_df["PHQ9_pred"] - residuals_df["PHQ9_actual"]
ax.bar(range(len(residuals_df)), resid, color=colors, edgecolor="black")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(residuals_df)))
ax.set_xticklabels(
    [f"{sid}\n({g}, PHQ={p})" for sid, g, p in
     zip(residuals_df.index, residuals_df["group"], residuals_df["PHQ9_actual"])],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylabel("Residual  (predicted − actual)")
rmse_test = float(np.sqrt(((y_test - y_pred_test) ** 2).mean()))
ax.set_title(f"Per-test-subject residuals   (test RMSE = {rmse_test:.2f})")
ax.grid(alpha=0.3, axis="y")
ax.legend(handles=[Patch(facecolor="tab:red", label="MDD"),
                   Patch(facecolor="tab:blue", label="HC")],
          loc="best")

plt.tight_layout()
plt.show()


In [ ]:
# CELL 9 — Helper: fit LOSO Elastic Net on a subset of audio files.
#
# For each grouping below we pick a subset of the 29 audio files
# (by valence or by task), compute the per-subject 88-dim mean over
# ONLY those files, and run the same LOSO Elastic Net pipeline as
# Cell 6 on the SAME 80/20 split from Cell 5. Reusing idx_train /
# idx_test lets us compare groupings like-for-like on identical test
# subjects (differences in RMSE reflect the grouping, not split noise).

def fit_subset_loso_enet(file_numbers, label):
    """Per-subject mean over `file_numbers`, LOSO ElasticNetCV on the
    training subjects from Cell 5, predict the same test subjects.
    Returns a dict of predictions + chosen hyperparameters + metrics."""

    frames = [func_dfs[fn] for fn in file_numbers if fn in func_dfs]
    stacked_sub = pd.concat(frames, axis=0)
    subj_mean = stacked_sub.groupby(level=0).mean()
    subj_mean = subj_mean.loc[subject_info.index]

    X_tr = subj_mean.loc[idx_train].values
    X_te = subj_mean.loc[idx_test].values

    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    X_te_s = scaler.transform(X_te)

    loo_local = LeaveOneOut()
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
        alphas=np.logspace(-3, 1, 40),
        cv=loo_local, max_iter=20000, n_jobs=-1,
        random_state=RANDOM_STATE,
    ).fit(X_tr_s, y_train)

    final = ElasticNet(
        alpha=enet.alpha_, l1_ratio=enet.l1_ratio_,
        max_iter=20000, random_state=RANDOM_STATE,
    )
    loso_train = cross_val_predict(final, X_tr_s, y_train, cv=loo_local)
    y_pred_te = enet.predict(X_te_s)

    return {
        "label": label,
        "n_files": len(file_numbers),
        "file_numbers": file_numbers,
        "alpha": float(enet.alpha_),
        "l1_ratio": float(enet.l1_ratio_),
        "n_nonzero": int((enet.coef_ != 0).sum()),
        "coef": enet.coef_,
        "train_loso_pred": loso_train,
        "test_pred": y_pred_te,
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, y_pred_te))),
        "test_mae": float(mean_absolute_error(y_test, y_pred_te)),
        "test_r2": float(r2_score(y_test, y_pred_te)),
        "train_loso_rmse": float(np.sqrt(mean_squared_error(y_train, loso_train))),
        "train_loso_mae": float(mean_absolute_error(y_train, loso_train)),
    }

print("Helper defined: fit_subset_loso_enet(file_numbers, label)")


In [ ]:
# CELL 10 — Grouping 1: 3 models by valence.
# Positive / Neutral / Negative. Files with no valence (19 = passage
# reading, 29 = TAT) are excluded per the user spec.

VALENCES = ["positive", "neutral", "negative"]

valence_file_groups = {
    v: audio_file_map.loc[audio_file_map["valence"] == v, "file_number"].tolist()
    for v in VALENCES
}
print("Valence → file numbers:")
for v, files in valence_file_groups.items():
    print(f"  {v:9s} ({len(files)} files): {files}")

valence_results = {
    v: fit_subset_loso_enet(files, f"valence={v}")
    for v, files in valence_file_groups.items()
}

valence_summary = pd.DataFrame({
    v: {
        "n_files": r["n_files"],
        "alpha": round(r["alpha"], 4),
        "l1_ratio": r["l1_ratio"],
        "n_nonzero": r["n_nonzero"],
        "train_LOSO_RMSE": round(r["train_loso_rmse"], 3),
        "train_LOSO_MAE": round(r["train_loso_mae"], 3),
        "test_RMSE": round(r["test_rmse"], 3),
        "test_MAE": round(r["test_mae"], 3),
        "test_R2": round(r["test_r2"], 3),
    } for v, r in valence_results.items()
}).T
print("\nValence model summary:")
print(valence_summary)


In [ ]:
# CELL 11 — Valence models — visualization.
# Top row: predicted-vs-actual PHQ-9 for each valence (same style as Cell 8
# left panel). Bottom: grouped bar chart of train LOSO-OOF RMSE and test
# RMSE per valence, with the mean-predictor baseline as a horizontal
# reference.

fig, axes = plt.subplots(
    2, 3, figsize=(16, 10),
    gridspec_kw={"height_ratios": [3, 2]},
)

for col, v in enumerate(VALENCES):
    r = valence_results[v]
    ax = axes[0, col]
    train_is_mdd = (grp_train == "MDD")
    test_is_mdd = (grp_test == "MDD")

    ax.scatter(y_train[~train_is_mdd], r["train_loso_pred"][~train_is_mdd],
               s=40, alpha=0.5, c="tab:blue", marker="o",
               label="Train HC (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_train[train_is_mdd], r["train_loso_pred"][train_is_mdd],
               s=40, alpha=0.5, c="tab:red", marker="o",
               label="Train MDD (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_test[~test_is_mdd], r["test_pred"][~test_is_mdd],
               s=180, alpha=0.95, c="tab:blue", marker="*",
               edgecolors="black", linewidths=1.1,
               label="Test HC" if col == 0 else None)
    ax.scatter(y_test[test_is_mdd], r["test_pred"][test_is_mdd],
               s=180, alpha=0.95, c="tab:red", marker="*",
               edgecolors="black", linewidths=1.1,
               label="Test MDD" if col == 0 else None)

    lims = [-2, 28]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.axvspan(7, 11, color="gray", alpha=0.12)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual PHQ-9")
    if col == 0:
        ax.set_ylabel("Predicted PHQ-9")
        ax.legend(loc="lower right", fontsize=8)
    ax.set_title(f"{v.capitalize()}\n(test RMSE = {r['test_rmse']:.2f},  n_files = {r['n_files']})")
    ax.grid(alpha=0.3)

# Merge bottom row into a single wide axes for the bar chart
gs = axes[1, 0].get_gridspec()
for ax in axes[1, :]:
    ax.remove()
ax_bar = fig.add_subplot(gs[1, :])

xpos = np.arange(len(VALENCES))
train_rmse = [valence_results[v]["train_loso_rmse"] for v in VALENCES]
test_rmse = [valence_results[v]["test_rmse"] for v in VALENCES]
w = 0.35
ax_bar.bar(xpos - w/2, train_rmse, w, label="Train LOSO-OOF RMSE",
           color="tab:gray", edgecolor="black")
ax_bar.bar(xpos + w/2, test_rmse, w, label="Test RMSE",
           color="tab:orange", edgecolor="black")
for i, (tr, te) in enumerate(zip(train_rmse, test_rmse)):
    ax_bar.text(i - w/2, tr + 0.05, f"{tr:.2f}", ha="center", fontsize=9)
    ax_bar.text(i + w/2, te + 0.05, f"{te:.2f}", ha="center", fontsize=9)

baseline_rmse = float(np.sqrt(((y_test - y_train.mean()) ** 2).mean()))
ax_bar.axhline(baseline_rmse, color="black", linestyle="--", alpha=0.6,
               label=f"Mean-predictor baseline ({baseline_rmse:.2f})")

ax_bar.set_xticks(xpos)
ax_bar.set_xticklabels([v.capitalize() for v in VALENCES])
ax_bar.set_ylabel("RMSE")
ax_bar.set_title("RMSE comparison across valence models")
ax_bar.legend(loc="best")
ax_bar.grid(alpha=0.3, axis="y")

fig.suptitle("Grouping 1 — PHQ-9 prediction by valence",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 12 — Grouping 2: 4 models by task.
# Tasks (from audio_file_map): Interview, Passage Reading, Word Reading,
# Picture Description.
#
# NOTE: Passage Reading has only 1 file per subject (file 19). "Per-subject
# mean over 1 file" is just that single file — noisier than tasks with 6+
# files averaged. Flag when interpreting its RMSE.

TASKS = audio_file_map["task"].unique().tolist()

task_file_groups = {
    t: audio_file_map.loc[audio_file_map["task"] == t, "file_number"].tolist()
    for t in TASKS
}
print("Task → file numbers:")
for t, files in task_file_groups.items():
    print(f"  {t:22s} ({len(files):2d} files): {files}")

task_results = {
    t: fit_subset_loso_enet(files, f"task={t}")
    for t, files in task_file_groups.items()
}

task_summary = pd.DataFrame({
    t: {
        "n_files": r["n_files"],
        "alpha": round(r["alpha"], 4),
        "l1_ratio": r["l1_ratio"],
        "n_nonzero": r["n_nonzero"],
        "train_LOSO_RMSE": round(r["train_loso_rmse"], 3),
        "train_LOSO_MAE": round(r["train_loso_mae"], 3),
        "test_RMSE": round(r["test_rmse"], 3),
        "test_MAE": round(r["test_mae"], 3),
        "test_R2": round(r["test_r2"], 3),
    } for t, r in task_results.items()
}).T
print("\nTask model summary:")
print(task_summary)


In [ ]:
# CELL 13 — Task models — visualization.
# Top row: predicted-vs-actual PHQ-9 for each task (same style as Cell 8
# and Cell 11). Bottom: bar chart of train LOSO-OOF RMSE and test RMSE
# per task, with the mean-predictor baseline as a horizontal reference.

n_tasks = len(TASKS)
fig, axes = plt.subplots(
    2, n_tasks, figsize=(5 * n_tasks, 10),
    gridspec_kw={"height_ratios": [3, 2]},
)

for col, t in enumerate(TASKS):
    r = task_results[t]
    ax = axes[0, col]
    train_is_mdd = (grp_train == "MDD")
    test_is_mdd = (grp_test == "MDD")

    ax.scatter(y_train[~train_is_mdd], r["train_loso_pred"][~train_is_mdd],
               s=35, alpha=0.5, c="tab:blue", marker="o",
               label="Train HC (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_train[train_is_mdd], r["train_loso_pred"][train_is_mdd],
               s=35, alpha=0.5, c="tab:red", marker="o",
               label="Train MDD (LOSO-OOF)" if col == 0 else None)
    ax.scatter(y_test[~test_is_mdd], r["test_pred"][~test_is_mdd],
               s=170, alpha=0.95, c="tab:blue", marker="*",
               edgecolors="black", linewidths=1.0,
               label="Test HC" if col == 0 else None)
    ax.scatter(y_test[test_is_mdd], r["test_pred"][test_is_mdd],
               s=170, alpha=0.95, c="tab:red", marker="*",
               edgecolors="black", linewidths=1.0,
               label="Test MDD" if col == 0 else None)

    lims = [-2, 28]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.axvspan(7, 11, color="gray", alpha=0.12)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual PHQ-9")
    if col == 0:
        ax.set_ylabel("Predicted PHQ-9")
        ax.legend(loc="lower right", fontsize=7)
    ax.set_title(f"{t}\n(test RMSE = {r['test_rmse']:.2f},  n_files = {r['n_files']})",
                 fontsize=10)
    ax.grid(alpha=0.3)

gs = axes[1, 0].get_gridspec()
for ax in axes[1, :]:
    ax.remove()
ax_bar = fig.add_subplot(gs[1, :])

xpos = np.arange(n_tasks)
train_rmse = [task_results[t]["train_loso_rmse"] for t in TASKS]
test_rmse = [task_results[t]["test_rmse"] for t in TASKS]
w = 0.35
ax_bar.bar(xpos - w/2, train_rmse, w, label="Train LOSO-OOF RMSE",
           color="tab:gray", edgecolor="black")
ax_bar.bar(xpos + w/2, test_rmse, w, label="Test RMSE",
           color="tab:orange", edgecolor="black")
for i, (tr, te) in enumerate(zip(train_rmse, test_rmse)):
    ax_bar.text(i - w/2, tr + 0.05, f"{tr:.2f}", ha="center", fontsize=9)
    ax_bar.text(i + w/2, te + 0.05, f"{te:.2f}", ha="center", fontsize=9)

baseline_rmse = float(np.sqrt(((y_test - y_train.mean()) ** 2).mean()))
ax_bar.axhline(baseline_rmse, color="black", linestyle="--", alpha=0.6,
               label=f"Mean-predictor baseline ({baseline_rmse:.2f})")

ax_bar.set_xticks(xpos)
ax_bar.set_xticklabels(TASKS, rotation=15, ha="right")
ax_bar.set_ylabel("RMSE")
ax_bar.set_title("RMSE comparison across task models")
ax_bar.legend(loc="best")
ax_bar.grid(alpha=0.3, axis="y")

fig.suptitle("Grouping 2 — PHQ-9 prediction by task",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
